# Checkpoint 47: Current-State Dashboard Validation

This notebook reviews the generated dashboard population, frozen human-review plan, timeline metadata, and validation checks. Run `scripts/run_checkpoint47.ps1` first.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DASHBOARD = PROCESSED / 'dashboard'

risk = pd.read_csv(DASHBOARD / 'retention_risk_employees.csv')
policy = pd.read_csv(DASHBOARD / 'current_policy_summary.csv')
metadata = pd.read_csv(DASHBOARD / 'dashboard_metadata.csv')
validation = pd.read_csv(PROCESSED / 'dashboard_current_state_validation.csv')


In [ ]:
display(metadata.T)
display(policy)
display(validation)

In [ ]:
population_check = pd.DataFrame({
    'metric': [
        'Dashboard rows',
        'Unique employees',
        'Inactive employees displayed',
        'Selected for human review',
        'Automatic actions permitted',
    ],
    'value': [
        len(risk),
        risk['employee_id'].nunique(),
        risk['employment_status'].ne('Active').sum(),
        risk['selected_for_human_review'].sum(),
        risk['automatic_employment_action_permitted'].sum(),
    ],
})
display(population_check)

In [ ]:
department_plan = (
    risk.groupby('department_name', as_index=False)
    .agg(
        eligible_employees=('employee_id', 'count'),
        selected_for_human_review=('selected_for_human_review', 'sum'),
        average_probability=('attrition_probability', 'mean'),
    )
)
display(department_plan.sort_values('selected_for_human_review', ascending=False))